In [2]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
import joblib

In [3]:
train = pd.read_csv("../artifacts/train.csv")
validation = pd.read_csv("../artifacts/validation.csv")
test = pd.read_csv("../artifacts/test.csv")

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

Train shape: (69608, 18)
Validation shape: (14916, 18)
Test shape: (14917, 18)


2.Convert date to datetime/
We need order_purchase_timestamp to extract time features from it.

In [4]:
date_columns = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date"
]

for df in [train, validation, test]:
    for col in date_columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

3.Creating time features/
From the EDA we found that the month, day of the week, and time of purchase have differences in the rate of delay, so we will convert the purchase date into separate features.

In [5]:
for df in [train, validation, test]:
    df["purchase_year"] = df["order_purchase_timestamp"].dt.year
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_dayofweek"] = df["order_purchase_timestamp"].dt.dayofweek
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

4.Create the expected delivery /We have order_estimated_delivery_date, which is different information from the actual delivery date.

This feature is useful because it represents the number of days that were expected between purchase and delivery.

We do not use:
order_delivered_customer_date
Order_delivered_carrier_date here

because this information occurs after the order is placed and is not available at the time of the forecast.

In [6]:
for df in [train, validation, test]:
    df["estimated_delivery_days"] = (
        df["order_estimated_delivery_date"]
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / (24 * 60 * 60)

5.Determine the features we will use.

The primary reason for the delivery columns is to prevent data leakage; the features must be available at the moment of prediction.

In [7]:
feature_columns = [
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    "number_of_items",
    "total_price",
    "total_freight",
    "total_payment",
    "max_installments",
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour",
    "estimated_delivery_days"
]

6.Separate X and y

In [8]:
X_train = train[feature_columns].copy()
y_train = train["late_delivery"].copy()

X_validation = validation[feature_columns].copy()
y_validation = validation["late_delivery"].copy()

X_test = test[feature_columns].copy()
y_test = test["late_delivery"].copy()

In [9]:
print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_validation:", y_validation.shape)
print("y_test:", y_test.shape)

X_train: (69608, 13)
X_validation: (14916, 13)
X_test: (14917, 13)
y_train: (69608,)
y_validation: (14916,)
y_test: (14917,)


7.Identifying Numerical and Categorical Features

In [10]:
numerical_features = [
    "customer_zip_code_prefix",
    "number_of_items",
    "total_price",
    "total_freight",
    "total_payment",
    "max_installments",
    "purchase_year",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour",
    "estimated_delivery_days"
]

categorical_features = [
    "customer_city",
    "customer_state"
]

8.Handling missing values ​​in Numerical Features

The EDA showed that we have missing values ​​in some numeric variables. Therefore, we will use SimpleImputer.

In [11]:
numeric_imputer = SimpleImputer(strategy="median")

X_train_numeric = numeric_imputer.fit_transform(
    X_train[numerical_features]
)

X_validation_numeric = numeric_imputer.transform(
    X_validation[numerical_features]
)

X_test_numeric = numeric_imputer.transform(
    X_test[numerical_features]
)

9.Handling Missing Values ​​and Categorical Features Encoding

In [12]:
categorical_imputer = SimpleImputer(
    strategy="most_frequent"
)

X_train_categorical_imputed = categorical_imputer.fit_transform(
    X_train[categorical_features]
)

X_validation_categorical_imputed = categorical_imputer.transform(
    X_validation[categorical_features]
)

X_test_categorical_imputed = categorical_imputer.transform(
    X_test[categorical_features]
)

In [13]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

In [14]:
X_train_categorical = encoder.fit_transform(
    X_train_categorical_imputed
)

X_validation_categorical = encoder.transform(
    X_validation_categorical_imputed
)

X_test_categorical = encoder.transform(
    X_test_categorical_imputed
)

10.Combining Numerical + Categorical

In [15]:
from scipy import sparse

X_train_numeric_sparse = sparse.csr_matrix(
    X_train_numeric
)

X_validation_numeric_sparse = sparse.csr_matrix(
    X_validation_numeric
)

X_test_numeric_sparse = sparse.csr_matrix(
    X_test_numeric
)

In [16]:
X_train_final = sparse.hstack(
    [X_train_numeric_sparse, X_train_categorical],
    format="csr"
)

X_validation_final = sparse.hstack(
    [X_validation_numeric_sparse, X_validation_categorical],
    format="csr"
)

X_test_final = sparse.hstack(
    [X_test_numeric_sparse, X_test_categorical],
    format="csr"
)

11.Check the format of the Feature Tables

In [17]:
print("Final train features:", X_train_final.shape)
print("Final validation features:", X_validation_final.shape)
print("Final test features:", X_test_final.shape)

Final train features: (69608, 3818)
Final validation features: (14916, 3818)
Final test features: (14917, 3818)


12.Creating feature names

In [18]:
categorical_feature_names = encoder.get_feature_names_out(
    categorical_features
)

feature_names = (
    numerical_features
    + list(categorical_feature_names)
)

print("Number of final features:", len(feature_names))

Number of final features: 3818


13.Save the Transformers

In [19]:
artifacts_dir = Path("../artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    numeric_imputer,
    artifacts_dir / "numeric_imputer.pkl"
)

joblib.dump(
    categorical_imputer,
    artifacts_dir / "categorical_imputer.pkl"
)

joblib.dump(
    encoder,
    artifacts_dir / "onehot_encoder.pkl"
)

print("Transformers saved successfully.")

Transformers saved successfully.


14.Save Feature List

In [20]:
feature_list_path = artifacts_dir / "feature_list.txt"

with open(feature_list_path, "w", encoding="utf-8") as f:
    for feature in feature_names:
        f.write(feature + "\n")

print("Feature list saved successfully.")

Feature list saved successfully.


15.Save the Final Feature Tables

In [21]:
sparse.save_npz(
    artifacts_dir / "X_train.npz",
    X_train_final
)

sparse.save_npz(
    artifacts_dir / "X_validation.npz",
    X_validation_final
)

sparse.save_npz(
    artifacts_dir / "X_test.npz",
    X_test_final
)

In [22]:
y_train.to_csv(
    artifacts_dir / "y_train.csv",
    index=False
)

y_validation.to_csv(
    artifacts_dir / "y_validation.csv",
    index=False
)

y_test.to_csv(
    artifacts_dir / "y_test.csv",
    index=False
)

In [23]:
print("Feature engineering artifacts:")

for path in sorted(artifacts_dir.glob("*")):
    print(path)

Feature engineering artifacts:
..\artifacts\categorical_imputer.pkl
..\artifacts\eda_charts
..\artifacts\eda_findings_summary.txt
..\artifacts\feature_list.txt
..\artifacts\labeled_table.csv
..\artifacts\ml_table.csv
..\artifacts\numeric_imputer.pkl
..\artifacts\onehot_encoder.pkl
..\artifacts\test.csv
..\artifacts\train.csv
..\artifacts\validation.csv
..\artifacts\X_test.npz
..\artifacts\X_train.npz
..\artifacts\X_validation.npz
..\artifacts\y_test.csv
..\artifacts\y_train.csv
..\artifacts\y_validation.csv
